# एसआरए मल्टी-डोमेन एलएलएम ट्यूटोरियल

इस नोटबुक में, आप छोटे स्तर के एलएलएम में एसआरए की ``एक साथ कई डोमेन (कोड, गणित, टेक्स्ट) को सीखने'' की विशेषता का अनुभव करेंगे।
डेटासेट के रूप में रिपॉजिटरी में `डेटा/लैंग/` में तीन प्रकार के डेटा का उपयोग करके, हम कल्पना करेंगे कि एसआरए स्वचालित रूप से प्रत्येक डोमेन के लिए सिनैप्स को कैसे विभाजित (विशेषीकृत) करता है।

In [ ]:
# Colab環境でのみ実行してください（ローカル環境の場合はスキップ可）
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch tiktoken matplotlib seaborn

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

## 1. डेटासेट को लोड करना और टोकनाइज़ करना
`code.txt`, `math.txt`, `text.txt` को `data/lang/` निर्देशिका में लोड करें।

In [ ]:
import torch
import tiktoken
import sys

tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

domains = ["code", "math", "text"]
data = {}

base_dir = "." if 'google.colab' in sys.modules else ".."

for d in domains:
    path = f"{base_dir}/data/lang/{d}.txt"
    with open(path, "r", encoding="utf-8") as f:
        # 少しデータをカサ増しして学習しやすくする
        text = (f.read() + "\n") * 5
    tokens = tokenizer.encode(text, allowed_special="all")
    data[d] = torch.tensor(tokens, dtype=torch.long)
    print(f"Domain '{d}': {len(tokens)} tokens")

## 2. मॉडल का निर्माण
पहले की तरह, एक छोटे कॉन्फ़िगरेशन के साथ एक `MoESRAlangageModel` बनाएं।

In [ ]:
from sra_language_models import MoESRALanguageModel

dim = 128
layers = 2
num_synapses = 16
k = 2
syn_hidden = 256
max_seq_len = 64

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"使用デバイス: {device}")

model = MoESRALanguageModel(
    vocab_size=vocab_size,
    dim=dim,
    layers=layers,
    num_synapses=num_synapses,
    k=k,
    syn_hidden=syn_hidden,
    max_seq_len=max_seq_len
).to(device)

## 3. मल्टी-डोमेन लर्निंग लूप
प्रत्येक चरण में, हम यादृच्छिक रूप से एक डोमेन चुनते हैं और सीखने के लिए उस पाठ से मिनी-बैच बनाते हैं।

In [ ]:
import random
import torch.nn.functional as F
from sra_experiment import make_optimizer, load_balance_loss

def get_multidomain_batch(data_dict, batch_size, seq_len):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    batch_domains = []
    
    for i in range(batch_size):
        d = random.choice(domains)
        batch_domains.append(d)
        d_data = data_dict[d]
        max_start = len(d_data) - seq_len - 1
        start = random.randint(0, max(0, max_start))
        
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
        
    return x.to(device), y.to(device), batch_domains

# 学習パラメータ
batch_size = 32
steps = 400
lr = 5e-4
opt = make_optimizer(model, lr)

model.train()
for step in range(1, steps + 1):
    x, y, b_domains = get_multidomain_batch(data, batch_size, max_seq_len)
    
    logits, router_logits = model(x, dense=False)
    
    ce_loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    lb_loss = load_balance_loss(router_logits)
    loss = ce_loss + 0.1 * lb_loss
    
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    
    if step % 50 == 0:
        print(f"Step {step}/{steps} | Loss: {loss.item():.4f}")

## 4. डोमेन द्वारा रूटिंग विज़ुअलाइज़ेशन
प्रशिक्षित मॉडल में प्रत्येक डोमेन से नमूना पाठ इनपुट करें, और फिर समग्र रूप से समग्र रूप से उपयोग किए जाने वाले सिनेप्स का विश्लेषण करें। आप एसआरए की "विशेषज्ञता" की जांच कर सकते हैं।

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sra_experiment import usage_stats

model.eval()
domain_usages = {}

with torch.no_grad():
    for d in domains:
        # 評価用に数バッチ回して使用状況を集計
        total_usage = None
        for _ in range(5):
            x, y, _ = get_multidomain_batch({d: data[d]}, batch_size, max_seq_len)
            _, router_logits = model(x)
            
            u = usage_stats(router_logits) # shape: (num_synapses,)
            total_usage = u if total_usage is None else total_usage + u
            
        # 正規化して保存
        domain_usages[d] = (total_usage / total_usage.sum()).cpu().numpy()

# ヒートマップ描画 (ドメイン x シナプス)
usage_matrix = np.array([domain_usages[d] for d in domains])

plt.figure(figsize=(10, 4))
sns.heatmap(usage_matrix, cmap="Blues", annot=True, fmt=".2f", yticklabels=domains)
plt.title("Synapse Usage per Domain")
plt.xlabel("Synapse ID")
plt.ylabel("Domain")
plt.show()